In [ ]:
import pandas as pd

In [ ]:
bd = pd.read_csv("cha_bd.bed", sep="\t",header=None)
bd.columns = ["chrom", "start", "end"]
bd

In [ ]:
wt = pd.read_csv("../../../IS/WT_is.tsv", sep="\t")
wt_final = wt[wt["is_bad_bin"]==False]

In [ ]:
import bioframe as bf
import bioframe.vis

olwt = bf.overlap(bd, wt_final, how="inner", suffixes=('_1','_2'))
olwt_final = olwt.groupby(["chrom_1","start_1","end_1"])["log2_insulation_score_1000000_2"].apply(list).reset_index(name="wt_is")

In [ ]:
pi = pd.read_csv("../../../IS/PI_is.tsv", sep="\t")
pi_final = pi[pi["is_bad_bin"]==False]

In [ ]:
olpi = bf.overlap(bd, pi_final, how="inner", suffixes=('_1','_2'))
olpi_final = olpi.groupby(["chrom_1","start_1","end_1"])["log2_insulation_score_1000000_2"].apply(list).reset_index(name="pi_is")

In [ ]:
out = pd.merge(olwt_final, olpi_final)
out

In [ ]:
def c_p():
    from scipy.stats import ttest_ind
    p = []
    for _, row in out.iterrows():
        stat,pvalue = ttest_ind(row['wt_is'], row['pi_is'])
        p.append(pvalue)
    return p

In [ ]:
def Average(lst):
    return sum(lst) / len(lst)

def c_fc():
    from scipy.stats import ttest_ind
    fc = []
    for _, row in out.iterrows():
        f = Average(row['wt_is'])/Average(row['pi_is'])
        fc.append(f)
    return fc

In [ ]:
p=c_p()
out["p-value"] = p
fc=c_fc()
out["fc"] = fc
out

In [ ]:
bd_change = out[(out["p-value"]<0.05)&(out["fc"]>1)]
bd_change.to_csv("bd_change.bed", sep="\t", index=False)